# 🚦 ITMS — Intelligent Traffic Management System (Kaggle Edition v2)
## Iqra University | BS Computer Science | AI Project

Runs **unattended in the background** on Kaggle, **resumes** after an interruption, and **tests itself in the first ~30 seconds** so a bug can't cost you hours.

---
## ▶️ Run it in the background ("off mode")
1. **Session options** (right panel): Accelerator = **GPU** (optional — the code falls back to CPU by itself), Internet = **ON** only if you want the optional YOLO demo (everything else works offline).
2. **Save Version → Save & Run All (Commit)**. Close the browser — Kaggle keeps running it.
3. Download `models/`, `results/`, `data/` from the version's **Output** tab.

## 🔄 If a run stops early
**Add Input → Notebook Output Files →** pick your previous version → **Save & Run All** again. Progress is restored automatically (look for `🔄 Restored` / `🔄 Resuming` in the log). To ignore old progress and start over, set `FORCE_FRESH = True` in the setup cell.

## 🛠️ What is different from v1 (and why)

| # | Problem found | Fix |
|---|---|---|
| 1 | **DQN environment was broken:** green time cost nothing, so every green ≥ 30 s gave the same reward (I measured it) — the agent could not learn timing | Rebuilt `TrafficEnv`: time passes, vehicles arrive during the green, 4 s lost time per phase change. Now short greens waste capacity and long greens make other approaches wait → a real trade-off |
| 2 | Step-9 pipeline fed the DQN a different state layout than it was trained on | One shared `make_state()` used by training **and** the pipeline |
| 3 | Step-9 fed the LSTM raw counts + zeros (it was trained on normalised features) | Pipeline builds real feature rows with the fitted scaler and converts predictions back to vehicles |
| 4 | Dashboard numbers (45 s→32 s, 800→950 …) were hard-coded | Everything on the dashboard is **measured**: DQN vs every fixed timing on identical random seeds |
| 5 | Scaler was fitted on rows that included LSTM test data (leakage) | Time-based 80/20 split per sensor; scaler fitted on train rows only |
| 6 | LSTM "MAE < 10 vehicles" was checked on normalised numbers | Metrics reported in real vehicles as well |
| 7 | DQN could end on a bad final policy (DQN is noisy) | Periodic validation; the **best** checkpoint is the one used |
| 8 | Possible divergence / NaN | Reward scaling, Huber loss, gradient clipping, Double-DQN, NaN guard (never overwrites a good checkpoint) |
| 9 | Resuming with changed code could crash or silently mix versions | Checkpoints carry a signature; incompatible ones are ignored with a warning |
| 10 | Half-written checkpoint if Kaggle killed the run mid-save | Atomic save (write to temp folder, then swap) with fallback to the previous one |
| 11 | `pip install` could break TensorFlow mid-run; YOLO could hang offline | YOLO moved to the very end, internet pre-checked, install time-limited, never fatal |
| 12 | A GPU incompatibility would show up after ~15 min | Smoke tests (LSTM + DQN + checkpoint round-trip) run first; GPU falls back to CPU automatically; DQN runs on CPU (faster for tiny networks) |

## STEP 1 — Setup (imports, folders, seeds, time guard, restore old progress)

In [ ]:
# STEP 1: setup
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import glob, json, time, random, shutil, warnings, tempfile
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import Sequential, layers, callbacks
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')
try:
    for _g in tf.config.list_physical_devices('GPU'):
        tf.config.experimental.set_memory_growth(_g, True)
except Exception:
    pass

FORCE_FRESH = False      # True = ignore any old checkpoints and start from scratch

SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

# time guard: Kaggle commit runs are killed after 12 h -> stop cleanly before that
NB_START        = time.time()
TIME_BUDGET_SEC = 10.5 * 3600
def elapsed_min(): return (time.time() - NB_START) / 60
def out_of_time(): return (time.time() - NB_START) > TIME_BUDGET_SEC

WORK        = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
DATA_DIR    = f'{WORK}/data'
MODELS_DIR  = f'{WORK}/models'
RESULTS_DIR = f'{WORK}/results'
CKPT_DIR    = f'{WORK}/ckpt'

if FORCE_FRESH:
    for d in (CKPT_DIR, MODELS_DIR):
        shutil.rmtree(d, ignore_errors=True)
for d in (DATA_DIR, MODELS_DIR, RESULTS_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)

def restore_previous_run():
    # copy checkpoints / finished models from a previous version added via "Add Input"
    patterns = ['/kaggle/input/**/ckpt/dqn_latest', '/kaggle/input/**/ckpt/dqn_old',
                '/kaggle/input/**/ckpt/lstm_backup', '/kaggle/input/**/models/lstm_final.keras']
    done = set()
    for pat in patterns:
        for hit in glob.glob(pat, recursive=True):
            src = os.path.dirname(hit)
            if src in done:
                continue
            dst = f'{WORK}/{os.path.basename(src)}'
            try:
                shutil.copytree(src, dst, dirs_exist_ok=True)
                done.add(src)
                print(f'🔄 Restored previous progress: {src}  →  {dst}')
            except Exception as e:
                print(f'⚠️ Could not restore {src}: {e}')
    if not done:
        print('ℹ️  No previous checkpoints found - starting fresh.')

if FORCE_FRESH:
    print('ℹ️  FORCE_FRESH=True - ignoring any previous progress.')
else:
    restore_previous_run()

print('\nEnvironment')
print('=' * 45)
print(f'  TensorFlow : {tf.__version__}')
print(f'  NumPy      : {np.__version__}')
print(f'  Pandas     : {pd.__version__}')
print(f'  GPU        : {len(tf.config.list_physical_devices("GPU"))} found')
print(f'  Work dir   : {WORK}')
print('=' * 45)

## STEP 2 — Definitions: LSTM builder, traffic environment, DQN agent, checkpoint helpers
Everything that can break is defined **here, first**, and smoke-tested in Step 3 before any long computation starts.

**Traffic environment (v2).** 4 approaches served one after another. Choosing a green duration `g`:
* the served approach discharges up to `0.5 veh/s × g`;
* a phase costs `g + 4 s` (4 s yellow/all-red lost time) during which **every** approach receives new vehicles;
* reward = vehicles cleared − (vehicle-seconds spent waiting) / 100;
* an episode is one simulated hour; demand level and per-approach imbalance are random every episode.

Short greens waste lost time, long greens make the other three approaches wait — so the best duration depends on the queues, which is exactly what the agent must learn.

In [ ]:
# STEP 2: definitions
# ------------------------------------------------------------------ LSTM
LOOKBACK = 30
FORECAST = 15
FEATURES = ['vehicle_count', 'flow_rate', 'occupancy', 'average_speed',
            'queue_length', 'congestion_level',
            'hour_sin', 'hour_cos', 'day_sin', 'day_cos']
TARGET   = 'vehicle_count'

def build_lstm(input_shape, output_steps):
    model = Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(128, return_sequences=True), layers.Dropout(0.2),
        layers.LSTM(128, return_sequences=True), layers.Dropout(0.2),
        layers.LSTM(64,  return_sequences=False), layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(output_steps),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss='mse', metrics=['mae'])
    return model

def lstm_callbacks(backup_dir, csv_path, best_path):
    try:
        backup = callbacks.BackupAndRestore(backup_dir=backup_dir)
    except TypeError:
        backup = callbacks.BackupAndRestore(backup_dir)
    return [
        callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
        callbacks.ModelCheckpoint(best_path, monitor='val_loss', save_best_only=True, verbose=0),
        backup,                                   # resume after a crash
        callbacks.CSVLogger(csv_path, append=True),
    ]

# ------------------------------------------------------------------ Traffic environment (v2)
ACTIONS = [20, 30, 40, 50, 60, 70, 80, 90, 120]    # green durations (seconds)

def make_state(queues, phase, last_green, progress):
    # Network input. The queue of the approach being served is ALWAYS first (rotated),
    # which makes "how long should this green be?" much easier to learn.
    q = np.roll(np.asarray(queues, dtype=np.float32), -int(phase)) / 50.0
    return np.array([*q, phase / 3.0, last_green / 120.0, progress], dtype=np.float32)

class TrafficEnv:
    SAT_FLOW   = 0.5      # veh/s discharged during green (~1800 veh/h)
    LOST_TIME  = 4.0      # s lost at every phase change
    BASE_RATE  = 0.08     # veh/s arriving per approach at demand multiplier 1.0
    BIAS       = np.array([1.0, 0.9, 1.1, 0.85])
    HORIZON    = 3600.0   # one simulated hour
    MAX_Q      = 200.0
    N_ACTIONS  = len(ACTIONS)
    STATE_SIZE = 7

    def __init__(self, seed=None):
        self.rng = np.random.RandomState(seed)
        self.reset()

    def reset(self):
        self.t, self.phase, self.last_green = 0.0, 0, 0.0
        self.demand = self.rng.uniform(0.6, 1.5) * self.BIAS * self.rng.lognormal(0, 0.15, 4)
        self.queues = self.rng.uniform(5, 20, 4)
        self.stats  = dict(queue_seconds=0.0, arrived=0.0, cleared=0.0, dropped=0.0, steps=0)
        return self._state()

    def _state(self):
        return make_state(self.queues, self.phase, self.last_green, self.t / self.HORIZON)

    def step(self, action_idx):
        green = float(ACTIONS[action_idx]); T = green + self.LOST_TIME
        q_before = self.queues.sum()
        cleared  = min(self.queues[self.phase], self.SAT_FLOW * green)
        self.queues[self.phase] -= cleared
        arrivals = self.rng.poisson(self.BASE_RATE * self.demand * T).astype(np.float64)
        new_q    = self.queues + arrivals
        dropped  = np.maximum(new_q - self.MAX_Q, 0).sum()
        self.queues = np.minimum(new_q, self.MAX_Q)
        delay  = 0.5 * (q_before + self.queues.sum()) * T       # vehicle-seconds waiting
        reward = cleared - delay / 100.0
        s = self.stats
        s['queue_seconds'] += delay; s['arrived'] += arrivals.sum()
        s['cleared'] += cleared;     s['dropped'] += dropped; s['steps'] += 1
        self.t += T; self.phase = (self.phase + 1) % 4; self.last_green = green
        return self._state(), float(reward), self.t >= self.HORIZON

def evaluate(policy, seeds):
    # Mean [reward, avg wait per vehicle (s), throughput (veh/h), avg total queue] over episodes.
    rows = []
    for sd in seeds:
        e = TrafficEnv(seed=sd); s = e.reset(); R = 0.0
        while True:
            s, r, d = e.step(policy(s, e)); R += r
            if d: break
        st = e.stats
        rows.append([R, st['queue_seconds'] / max(st['arrived'], 1.0),
                     st['cleared'] / (e.t / 3600.0), st['queue_seconds'] / e.t])
    return np.array(rows).mean(axis=0)

def heuristic_policy(state, env):
    # rule-based reference: green just long enough to clear the served queue
    need = env.queues[env.phase] / env.SAT_FLOW + 2.0
    return int(np.argmin([abs(g - need) for g in ACTIONS]))

# ------------------------------------------------------------------ DQN agent
CPU = '/CPU:0'     # the network is tiny: CPU is faster than GPU (no per-step transfer latency)

class DQNAgent:
    def __init__(self, state_size=7, action_size=9, gamma=0.95, lr=1e-3, buffer_size=10_000,
                 batch_size=32, warmup=500, target_sync_steps=500, reward_scale=0.05):
        self.state_size, self.action_size = state_size, action_size
        self.gamma, self.batch_size, self.warmup = gamma, batch_size, warmup
        self.target_sync_steps, self.reward_scale = target_sync_steps, reward_scale
        self.epsilon, self.eps_min, self.eps_decay = 1.0, 0.01, 0.989     # decay once per EPISODE
        self.buffer_size = buffer_size
        self.S  = np.zeros((buffer_size, state_size), np.float32)
        self.A  = np.zeros(buffer_size, np.int32)
        self.R  = np.zeros(buffer_size, np.float32)
        self.S2 = np.zeros((buffer_size, state_size), np.float32)
        self.D  = np.zeros(buffer_size, np.float32)
        self.ptr, self.size, self.train_steps = 0, 0, 0
        with tf.device(CPU):
            self.model  = self._build()
            self.target = self._build()
            self.update_target()
            self.opt = tf.keras.optimizers.Adam(lr, clipnorm=1.0)
            try:
                self.opt.build(self.model.trainable_variables)
            except Exception:
                pass

    def _build(self):
        return tf.keras.Sequential([
            layers.Input(shape=(self.state_size,)),
            layers.Dense(128, activation='relu'),
            layers.Dense(128, activation='relu'),
            layers.Dense(64,  activation='relu'),
            layers.Dense(self.action_size),
        ])

    @tf.function
    def _q_values(self, s):
        return self.model(s, training=False)

    @tf.function
    def _train_step(self, S, A, R, S2, D):
        # Double DQN target + Huber loss (robust to large rewards)
        a_star = tf.argmax(self.model(S2, training=False), axis=1, output_type=tf.int32)
        q_next = tf.reduce_sum(self.target(S2, training=False) * tf.one_hot(a_star, self.action_size), axis=1)
        y = tf.stop_gradient(R + self.gamma * q_next * (1.0 - D))
        with tf.GradientTape() as tape:
            q_a  = tf.reduce_sum(self.model(S, training=True) * tf.one_hot(A, self.action_size), axis=1)
            err  = y - q_a
            ae   = tf.abs(err)
            quad = tf.minimum(ae, 1.0)
            loss = tf.reduce_mean(0.5 * tf.square(quad) + (ae - quad))
        grads = tape.gradient(loss, self.model.trainable_variables)
        self.opt.apply_gradients(zip(grads, self.model.trainable_variables))
        return loss

    def q_values(self, state):
        with tf.device(CPU):
            x = tf.convert_to_tensor(np.asarray(state, np.float32)[np.newaxis])
            return self._q_values(x).numpy()[0]

    def act(self, state, training=True):
        if training and random.random() < self.epsilon:
            return random.randrange(self.action_size)
        return int(np.argmax(self.q_values(state)))

    def remember(self, s, a, r, s2, done):
        i = self.ptr
        self.S[i], self.A[i], self.R[i] = s, a, r * self.reward_scale
        self.S2[i], self.D[i] = s2, float(done)
        self.ptr  = (self.ptr + 1) % self.buffer_size
        self.size = min(self.size + 1, self.buffer_size)

    def replay(self):
        if self.size < max(self.batch_size, self.warmup):
            return None
        idx = np.random.randint(0, self.size, self.batch_size)
        with tf.device(CPU):
            loss = self._train_step(self.S[idx], self.A[idx], self.R[idx], self.S2[idx], self.D[idx])
        self.train_steps += 1
        if self.train_steps % self.target_sync_steps == 0:
            self.update_target()
        return float(loss)

    def decay_epsilon(self):
        self.epsilon = max(self.eps_min, self.epsilon * self.eps_decay)

    def update_target(self):
        self.target.set_weights(self.model.get_weights())

    def save(self, path):
        self.model.save(path)

# ------------------------------------------------------------------ checkpoints (atomic + versioned)
SIGNATURE = {'env': 'v2', 'state': 7, 'actions': ACTIONS, 'net': [128, 128, 64],
             'gamma': 0.95, 'reward_scale': 0.05, 'double_dqn': True}

def save_checkpoint(agent, st, ckpt_dir=None):
    ckpt_dir = ckpt_dir or CKPT_DIR
    new, latest, old = (f'{ckpt_dir}/dqn_{n}' for n in ('new', 'latest', 'old'))
    shutil.rmtree(new, ignore_errors=True); os.makedirs(new)
    agent.model.save_weights(f'{new}/model.weights.h5')
    agent.target.save_weights(f'{new}/target.weights.h5')
    np.savez_compressed(f'{new}/buffer.npz', S=agent.S[:agent.size], A=agent.A[:agent.size],
                        R=agent.R[:agent.size], S2=agent.S2[:agent.size], D=agent.D[:agent.size],
                        ptr=agent.ptr, size=agent.size)
    with open(f'{new}/state.json', 'w') as f:
        json.dump({**st, 'epsilon': float(agent.epsilon), 'train_steps': int(agent.train_steps),
                   'signature': SIGNATURE}, f)
    shutil.rmtree(old, ignore_errors=True)               # swap: new -> latest (previous kept as fallback)
    if os.path.isdir(latest):
        os.replace(latest, old)
    os.replace(new, latest)
    shutil.rmtree(old, ignore_errors=True)

def load_checkpoint(agent, ckpt_dir=None):
    ckpt_dir = ckpt_dir or CKPT_DIR
    for name in ('latest', 'old'):
        d = f'{ckpt_dir}/dqn_{name}'
        if not os.path.exists(f'{d}/state.json'):
            continue
        try:
            with open(f'{d}/state.json') as f:
                st = json.load(f)
            if st.get('signature') != SIGNATURE:
                print(f'⚠️ Ignoring checkpoint {d}: made by a different version of the code.')
                continue
            agent.model.load_weights(f'{d}/model.weights.h5')
            agent.target.load_weights(f'{d}/target.weights.h5')
            buf = np.load(f'{d}/buffer.npz'); n = int(buf['size'])
            agent.S[:n], agent.A[:n], agent.R[:n] = buf['S'], buf['A'], buf['R']
            agent.S2[:n], agent.D[:n] = buf['S2'], buf['D']
            agent.size, agent.ptr = n, int(buf['ptr'])
            agent.epsilon, agent.train_steps = float(st['epsilon']), int(st['train_steps'])
            return st
        except Exception as e:
            print(f'⚠️ Could not read checkpoint {d} ({type(e).__name__}: {e})')
    return None

def save_best_weights(agent, ckpt_dir=None):
    ckpt_dir = ckpt_dir or CKPT_DIR
    tmp = f'{ckpt_dir}/dqn_best.tmp.weights.h5'
    agent.model.save_weights(tmp)
    os.replace(tmp, f'{ckpt_dir}/dqn_best.weights.h5')

print('✅ Definitions loaded')

## STEP 3 — Smoke tests (fail fast)
Runs a tiny LSTM fit with the real callbacks, a tiny DQN run, and a checkpoint save → load round-trip. Takes seconds. If anything is incompatible with this Kaggle image you find out **now**, not after an hour.

In [ ]:
# STEP 3: smoke tests
def lstm_smoke(device):
    with tempfile.TemporaryDirectory() as tmp, tf.device(device):
        m = build_lstm((LOOKBACK, len(FEATURES)), FORECAST)
        X = np.random.rand(16, LOOKBACK, len(FEATURES)).astype('float32')
        y = np.random.rand(16, FORECAST).astype('float32')
        m.fit(X, y, validation_data=(X, y), epochs=1, batch_size=8, verbose=0,
              callbacks=lstm_callbacks(f'{tmp}/backup', f'{tmp}/h.csv', f'{tmp}/best.keras'))
        m.save(f'{tmp}/m.keras')
        tf.keras.models.load_model(f'{tmp}/m.keras')

LSTM_DEVICE = '/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'
try:
    lstm_smoke(LSTM_DEVICE)
except Exception as e:
    if LSTM_DEVICE == '/CPU:0':
        raise
    print(f'⚠️ GPU test failed ({type(e).__name__}: {e})\n   → the LSTM will use the CPU instead.')
    LSTM_DEVICE = '/CPU:0'
    lstm_smoke(LSTM_DEVICE)
print(f'✅ LSTM smoke test passed on {LSTM_DEVICE}')

def dqn_smoke():
    a = DQNAgent(warmup=64); e = TrafficEnv(seed=1); s = e.reset(); losses = []
    for _ in range(150):
        act = a.act(s); s2, r, d = e.step(act); a.remember(s, act, r, s2, d)
        l = a.replay()
        if l is not None: losses.append(l)
        s = e.reset() if d else s2
    assert losses and all(np.isfinite(losses)), 'DQN produced no / non-finite loss'
    with tempfile.TemporaryDirectory() as tmp:
        save_checkpoint(a, {'episode': 1}, tmp)
        save_checkpoint(a, {'episode': 2}, tmp)          # exercises the atomic swap path
        b = DQNAgent()
        st = load_checkpoint(b, tmp)
        assert st is not None and st['episode'] == 2, 'checkpoint did not load'
        assert b.size == a.size and b.ptr == a.ptr
        assert np.allclose(a.q_values(s), b.q_values(s), atol=1e-5), 'restored weights differ'
dqn_smoke()
print('✅ DQN smoke test passed (train step, checkpoint save/load, restored weights identical)')

## STEP 4 — Generate Synthetic Karachi Traffic Data
Morning peak 7–9 AM, evening peak 4–7 PM, 1% emergency vehicles, 0.5% accidents. 43,200 rows (10 h × 3 intersections × 4 directions × 10 s). Seeded → identical on every run. The per-sensor reading logic is a function (`sample_reading`) so the Step-10 pipeline uses **exactly** the same sensor model.

In [ ]:
# STEP 4: generate synthetic Karachi traffic data
np.random.seed(SEED)

INTERSECTIONS = ['karachi_j1', 'karachi_j2', 'karachi_j3']
DIRECTIONS    = ['north', 'south', 'east', 'west']
STEP_SEC      = 10
SIM_HOURS     = 10

def traffic_flow(hour, direction):
    am = 80 * np.exp(-0.5 * ((hour - 8) / 0.8) ** 2)
    pm = 90 * np.exp(-0.5 * ((hour - 17) / 1.0) ** 2)
    base = max(am + pm, 15)
    bias = {'north': 1.0, 'south': 0.9, 'east': 1.1, 'west': 0.85}
    return max(0, int(base * bias[direction] + np.random.normal(0, 8)))

def sample_reading(hour, direction):
    # one detector reading for one approach (shared by data generation and the pipeline)
    count     = traffic_flow(hour, direction)
    occupancy = min(1.0, count / 120 + np.random.normal(0, 0.03))
    speed     = max(5, 25 - occupancy * 20 + np.random.normal(0, 2))
    queue     = max(0, int(count * 0.25 + np.random.normal(0, 2)))
    flow_rate = count * 0.8 + np.random.normal(0, 3)
    return {'vehicle_count': count, 'flow_rate': round(flow_rate, 2),
            'occupancy': round(max(0, occupancy), 4), 'average_speed': round(speed, 2),
            'queue_length': queue}

records = []
start_time = datetime(2024, 1, 15, 7, 0, 0)       # a Monday
for step in range(SIM_HOURS * 3600 // STEP_SEC):
    ts   = start_time + timedelta(seconds=step * STEP_SEC)
    hour = ts.hour + ts.minute / 60
    for intersection in INTERSECTIONS:
        for direction in DIRECTIONS:
            rd = sample_reading(hour, direction)
            records.append({
                'timestamp': ts.isoformat(), 'hour': ts.hour, 'minute': ts.minute,
                'day_of_week': ts.weekday(), 'intersection_id': intersection, 'direction': direction,
                **rd,
                'emergency': 1 if np.random.random() < 0.01 else 0,
                'accident':  1 if np.random.random() < 0.005 else 0,
            })

df_raw = pd.DataFrame(records)
df_raw.to_csv(f'{DATA_DIR}/raw_traffic_data.csv', index=False)

print('Raw Traffic Data Generated')
print('=' * 45)
print(f'  Total records     : {len(df_raw):,}')
print(f'  Intersections     : {df_raw["intersection_id"].nunique()}')
print(f'  Directions        : {df_raw["direction"].nunique()}')
print(f'  Time range        : {df_raw["timestamp"].iloc[0]} → {df_raw["timestamp"].iloc[-1]}')
print(f'  Emergency events  : {df_raw["emergency"].sum()}')
print(f'  Accident events   : {df_raw["accident"].sum()}')
print(f'  Avg vehicle count : {df_raw["vehicle_count"].mean():.1f}')
print(f'  Max vehicle count : {df_raw["vehicle_count"].max()}')
print('=' * 45)

## STEP 5 — Preprocess & Engineer Features
Time-based 80/20 split **per sensor** (first 80% of each sensor's timeline = train). The scaler is fitted on training rows only, so no test statistics leak into training.

In [ ]:
# STEP 5: preprocess and engineer features
df = df_raw.copy()

df['vehicle_count'] = df['vehicle_count'].clip(0, 500)
df['occupancy']     = df['occupancy'].clip(0, 1)
df['average_speed'] = df['average_speed'].clip(0, 50)
df['queue_length']  = df['queue_length'].clip(0, 200)
df['flow_rate']     = df['flow_rate'].clip(0, 600)

df['congestion_level'] = (df['occupancy'] * 0.4 + df['queue_length'] / 50 * 0.4 +
                          (1 - df['average_speed'] / 50) * 0.2).clip(0, 1)

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['day_sin']  = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['day_cos']  = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['is_peak']  = df['hour'].apply(lambda h: 1 if (7 <= h < 9 or 16 <= h < 19) else 0)

df = df.sort_values(['intersection_id', 'direction', 'timestamp']).reset_index(drop=True)
gkeys = ['intersection_id', 'direction']
for lag in [1, 2, 3, 6, 12]:
    df[f'count_lag_{lag}'] = df.groupby(gkeys)['vehicle_count'].shift(lag)

grp = df.groupby(gkeys)['vehicle_count']
for w in [3, 6, 12]:
    df[f'roll_mean_{w}'] = grp.transform(lambda x: x.rolling(w, min_periods=1).mean())
    df[f'roll_std_{w}']  = grp.transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))
    df[f'roll_max_{w}']  = grp.transform(lambda x: x.rolling(w, min_periods=1).max())

df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

# time-based split per sensor
pos  = df.groupby(gkeys).cumcount()
size = df.groupby(gkeys)['timestamp'].transform('size')
df['is_train'] = (pos < 0.8 * size).astype(int)

NUM_COLS = ['vehicle_count', 'flow_rate', 'occupancy', 'average_speed', 'queue_length', 'congestion_level']
scaler = StandardScaler().fit(df.loc[df['is_train'] == 1, NUM_COLS])      # train rows only

df_proc = df.copy()
df_proc[NUM_COLS] = scaler.transform(df[NUM_COLS])
df_proc.to_csv(f'{DATA_DIR}/processed_traffic_data.csv', index=False)

VC_MEAN, VC_STD = float(scaler.mean_[0]), float(scaler.scale_[0])     # to convert vehicle_count back
with open(f'{MODELS_DIR}/scaler.json', 'w') as f:
    json.dump({'columns': NUM_COLS, 'mean': scaler.mean_.tolist(), 'scale': scaler.scale_.tolist()}, f, indent=2)

print('Preprocessing Complete')
print('=' * 45)
print(f'  Original features  : {len(df_raw.columns)}')
print(f'  Engineered features: {len(df_proc.columns)}')
print(f'  Training rows      : {int(df_proc["is_train"].sum()):,}')
print(f'  Test rows          : {int((1 - df_proc["is_train"]).sum()):,}')
print(f'  Null values        : {int(df_proc.isnull().sum().sum())}')
print(f'  Rows dropped (lags): {len(df_raw) - len(df_proc)}')
print('=' * 45)

## STEP 6 — Plot Raw Traffic Patterns

In [ ]:
# STEP 6: visualise traffic patterns
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Karachi Traffic Data — Exploratory Analysis', fontsize=15, fontweight='bold')

ax = axes[0, 0]
hourly = df_raw.groupby('hour')['vehicle_count'].mean()
ax.bar(hourly.index, hourly.values, color='steelblue', edgecolor='white')
ax.axvspan(7, 9, alpha=0.2, color='red', label='AM Peak')
ax.axvspan(16, 19, alpha=0.2, color='orange', label='PM Peak')
ax.set_title('Average Vehicle Count by Hour'); ax.set_xlabel('Hour'); ax.set_ylabel('Vehicles')
ax.legend(); ax.grid(axis='y', alpha=0.3)

ax = axes[0, 1]
ax.hist(df_raw['vehicle_count'], bins=40, color='teal', edgecolor='white', alpha=0.8)
ax.set_title('Vehicle Count Distribution'); ax.set_xlabel('Vehicles'); ax.set_ylabel('Frequency')
ax.grid(alpha=0.3)

ax = axes[1, 0]
sample = df_raw.sample(2000, random_state=SEED)
sc = ax.scatter(sample['occupancy'], sample['average_speed'], c=sample['vehicle_count'],
                cmap='RdYlGn_r', alpha=0.5, s=10)
plt.colorbar(sc, ax=ax, label='Vehicle Count')
ax.set_title('Speed vs Occupancy'); ax.set_xlabel('Occupancy'); ax.set_ylabel('Speed (km/h)')

ax = axes[1, 1]
for jct in INTERSECTIONS:
    sub = df_raw[df_raw['intersection_id'] == jct].groupby('hour')['vehicle_count'].mean()
    ax.plot(sub.index, sub.values, marker='o', markersize=4, label=jct)
ax.set_title('Hourly Volume by Intersection'); ax.set_xlabel('Hour'); ax.set_ylabel('Vehicles')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/traffic_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Chart saved to {RESULTS_DIR}/traffic_analysis.png')

## STEP 7 — LSTM Traffic Predictor
Resumable (per-epoch backup). If `models/lstm_final.keras` already exists and has the right shape, training is skipped.

In [ ]:
# STEP 7a: sequence windows (junction j1, north approach)
def make_sequences(data, lookback, forecast, features, target):
    X, y = [], []
    vals, tgt = data[features].values, data[target].values
    for i in range(lookback, len(vals) - forecast):
        X.append(vals[i - lookback: i]); y.append(tgt[i: i + forecast])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

df_j1 = df_proc[(df_proc['intersection_id'] == 'karachi_j1') &
                (df_proc['direction'] == 'north')].reset_index(drop=True)
X_tr, y_tr = make_sequences(df_j1[df_j1['is_train'] == 1], LOOKBACK, FORECAST, FEATURES, TARGET)
X_te, y_te = make_sequences(df_j1[df_j1['is_train'] == 0], LOOKBACK, FORECAST, FEATURES, TARGET)

assert len(X_tr) > 100 and len(X_te) > 50, 'too few sequences - check the split'
assert np.isfinite(X_tr).all() and np.isfinite(y_tr).all(), 'NaN/inf in training sequences'

print('Sequence Shapes')
print(f'  X_train : {X_tr.shape}   →  (samples, timesteps, features)')
print(f'  y_train : {y_tr.shape}')
print(f'  X_test  : {X_te.shape}')
print(f'  y_test  : {y_te.shape}')

In [ ]:
# STEP 7b: train the LSTM (or load the finished one)
LSTM_FINAL = f'{MODELS_DIR}/lstm_final.keras'
LSTM_CSV   = f'{MODELS_DIR}/lstm_history.csv'
LSTM_BEST  = f'{MODELS_DIR}/lstm_best.keras'
lstm_model = None

if os.path.exists(LSTM_FINAL):
    try:
        m = tf.keras.models.load_model(LSTM_FINAL)
        with tf.device(LSTM_DEVICE):       # functional shape check - works on every Keras version
            probe = m.predict(np.zeros((1, LOOKBACK, len(FEATURES)), 'float32'), verbose=0)
        if probe.shape == (1, FORECAST):
            lstm_model = m
            print(f'✅ Found a finished LSTM ({LSTM_FINAL}) - loaded, training skipped.')
        else:
            print('⚠️ Saved LSTM has a different shape - retraining.')
    except Exception as e:
        print(f'⚠️ Could not load saved LSTM ({type(e).__name__}: {e}) - retraining.')

if lstm_model is None:
    backup_dir = f'{CKPT_DIR}/lstm_backup'
    if not (os.path.isdir(backup_dir) and os.listdir(backup_dir)) and os.path.exists(LSTM_CSV):
        os.remove(LSTM_CSV)                     # stale history from an unrelated run
    elif os.path.isdir(backup_dir) and os.listdir(backup_dir):
        print('🔄 Resuming LSTM training from the last completed epoch')
    with tf.device(LSTM_DEVICE):
        lstm_model = build_lstm((LOOKBACK, len(FEATURES)), FORECAST)
        lstm_model.summary()
        print('\nTraining LSTM ... (backed up every epoch)')
        lstm_model.fit(X_tr, y_tr, validation_data=(X_te, y_te), epochs=100, batch_size=32,
                       callbacks=lstm_callbacks(backup_dir, LSTM_CSV, LSTM_BEST), verbose=2)
    lstm_model.save(LSTM_FINAL)
    print(f'\n✅ LSTM training complete! Saved to {LSTM_FINAL}')

In [ ]:
# STEP 7c: evaluate the LSTM (normalised AND in real vehicles) and plot
with tf.device(LSTM_DEVICE):
    y_pred = lstm_model.predict(X_te, verbose=0)

mae_n  = mean_absolute_error(y_te.flatten(), y_pred.flatten())
rmse_n = float(np.sqrt(mean_squared_error(y_te.flatten(), y_pred.flatten())))
r2     = r2_score(y_te.flatten(), y_pred.flatten())
mae_v, rmse_v = mae_n * VC_STD, rmse_n * VC_STD          # back to vehicles
ok = mae_v < 10 and rmse_v < 15 and r2 > 0.85

print('LSTM Evaluation Results (test set = last 20% of the timeline)')
print('=' * 55)
print(f'  MAE   : {mae_v:7.2f} vehicles   (normalised {mae_n:.4f})   target < 10')
print(f'  RMSE  : {rmse_v:7.2f} vehicles   (normalised {rmse_n:.4f})   target < 15')
print(f'  R²    : {r2:7.4f}                                   target > 0.85')
print(f'  Status: {"✅ ALL TARGETS MET" if ok else "⚠️ Some targets not met"}')
print('=' * 55)

with open(f'{MODELS_DIR}/lstm_metrics.json', 'w') as f:
    json.dump({'mae_vehicles': round(float(mae_v), 3), 'rmse_vehicles': round(float(rmse_v), 3),
               'r2': round(float(r2), 4), 'mae_normalised': round(float(mae_n), 4),
               'rmse_normalised': round(rmse_n, 4)}, f, indent=2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LSTM Model — Training & Prediction Results', fontsize=13, fontweight='bold')
ax = axes[0]
if os.path.exists(LSTM_CSV):
    hist = pd.read_csv(LSTM_CSV)
    ax.plot(hist['loss'], label='Train Loss', color='steelblue')
    ax.plot(hist['val_loss'], label='Val Loss', color='orange')
ax.set_title('Loss Curve (MSE, normalised)'); ax.set_xlabel('Epoch'); ax.set_ylabel('MSE')
ax.legend(); ax.grid(alpha=0.3)
ax = axes[1]
idx = min(10, len(y_te) - 1)
ax.plot(y_te[idx] * VC_STD + VC_MEAN, label='Actual', color='steelblue', linewidth=2, marker='o', markersize=4)
ax.plot(y_pred[idx] * VC_STD + VC_MEAN, label='Predicted', color='red', linewidth=2,
        linestyle='--', marker='x', markersize=5)
ax.set_title('Sample 15-step forecast'); ax.set_xlabel('Step ahead (×10 s)'); ax.set_ylabel('Vehicles')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/lstm_results.png', dpi=150, bbox_inches='tight')
plt.show()

## STEP 8 — Train the DQN Signal Optimizer (fast, resumable, keeps the best policy)
* Checkpoint every 5 episodes (atomic). Resumes automatically.
* Every 25 episodes the greedy policy is scored on 10 fixed validation seeds; the **best** weights are kept and used afterwards.
* Stops safely on a NaN loss (the last good checkpoint is never overwritten) or when the time budget runs out.

In [ ]:
# STEP 8a: train the DQN
EPISODES    = 400
CKPT_EVERY  = 5
EVAL_EVERY  = 25
VAL_SEEDS   = list(range(9000, 9010))      # fixed validation scenarios (never used for training)

agent = DQNAgent()
st = load_checkpoint(agent)
if st is None:
    agent = DQNAgent()                      # fresh agent (in case a failed load left it half-restored)

rewards_per_episode, losses_per_episode, epsilon_per_episode, val_history = [], [], [], []
best_score, best_episode, start_ep = -1e18, 0, 1
if st:
    rewards_per_episode = st['rewards'];  losses_per_episode = st['losses']
    epsilon_per_episode = st['epsilons']; val_history = st['val_history']
    best_score, best_episode = st['best_score'], st['best_episode']
    start_ep = st['episode'] + 1
    print(f'🔄 Resuming DQN: {st["episode"]}/{EPISODES} episodes already done, continuing at {start_ep}')
else:
    print('Starting DQN training from episode 1')

def snapshot(ep):
    return {'episode': ep, 'rewards': rewards_per_episode, 'losses': losses_per_episode,
            'epsilons': epsilon_per_episode, 'val_history': val_history,
            'best_score': best_score, 'best_episode': best_episode}

if start_ep > EPISODES:
    print(f'✅ All {EPISODES} episodes were already completed - nothing to train.')

random.seed(SEED + start_ep); np.random.seed(SEED + start_ep)      # different random stream after a resume
env = TrafficEnv(seed=SEED + start_ep)
stopped_early = diverged = False

print(f'\n{"Episode":>8} {"Train reward":>13} {"Epsilon":>8} {"Val reward":>11} {"Best":>9} {"Elapsed":>9}')
print('-' * 66)
for ep in range(start_ep, EPISODES + 1):
    t0 = time.time()
    state, total_reward, ep_losses = env.reset(), 0.0, []
    while True:
        action = agent.act(state)
        nxt, reward, done = env.step(action)
        agent.remember(state, action, reward, nxt, done)
        loss = agent.replay()
        if loss is not None:
            if not np.isfinite(loss):
                diverged = True
                break
            ep_losses.append(loss)
        state, total_reward = nxt, total_reward + reward
        if done:
            break
    if diverged:
        print(f'\n❌ Loss became NaN/inf in episode {ep}. Stopping WITHOUT saving; the last good '
              'checkpoint is untouched. Re-run to continue from it.')
        break

    agent.decay_epsilon()
    rewards_per_episode.append(float(total_reward))
    losses_per_episode.append(float(np.mean(ep_losses)) if ep_losses else 0.0)
    epsilon_per_episode.append(float(agent.epsilon))

    val = None
    if ep % EVAL_EVERY == 0 or ep == EPISODES:
        val = float(evaluate(lambda s, e: agent.act(s, training=False), VAL_SEEDS)[0])
        val_history.append([ep, val])
        if val > best_score:
            best_score, best_episode = val, ep
            save_best_weights(agent)

    if ep == start_ep or ep % EVAL_EVERY == 0:
        print(f'{ep:>8} {np.mean(rewards_per_episode[-10:]):>13.1f} {agent.epsilon:>8.3f} '
              f'{(f"{val:.1f}" if val is not None else "-"):>11} {best_score if best_episode else float("nan"):>9.1f} '
              f'{elapsed_min():>7.1f}m' + (f'   (episode took {time.time()-t0:.1f}s)' if ep == start_ep else ''),
              flush=True)

    if ep % CKPT_EVERY == 0 or ep == EPISODES:
        save_checkpoint(agent, snapshot(ep))
    if out_of_time():
        save_checkpoint(agent, snapshot(ep))
        print(f'\n⏱️ Time budget reached after episode {ep}. Checkpoint saved - add this version as Input and re-run to continue.')
        stopped_early = True
        break

# use the BEST policy (not just the last one) for everything that follows
best_path = f'{CKPT_DIR}/dqn_best.weights.h5'
if os.path.exists(best_path):
    agent.model.load_weights(best_path)
    print(f'\n🏆 Using the best policy: episode {best_episode} (validation reward {best_score:.1f})')
else:
    print('\nℹ️ No validation checkpoint yet - using the latest weights.')

agent.save(f'{MODELS_DIR}/dqn_traffic_signal.keras')
with open(f'{MODELS_DIR}/dqn_metrics.json', 'w') as f:
    json.dump({'episodes_completed': len(rewards_per_episode), 'best_episode': best_episode,
               'best_validation_reward': round(float(best_score), 2) if best_episode else None,
               'final_epsilon': round(float(agent.epsilon), 4)}, f, indent=2)
print(f'{"✅ DQN training complete!" if not (stopped_early or diverged) else "⚠️ DQN stopped early."} '
      f'Model saved to {MODELS_DIR}/dqn_traffic_signal.keras')

In [ ]:
# STEP 8b: plot DQN training
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('DQN Training Results', fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(rewards_per_episode, alpha=0.35, color='steelblue', label='Episode reward (with exploration)')
ax.plot(pd.Series(rewards_per_episode).rolling(10).mean(), color='red', linewidth=2, label='10-episode average')
ax.set_title('Training reward'); ax.set_xlabel('Episode'); ax.set_ylabel('Reward'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
if val_history:
    ve, vv = zip(*val_history)
    ax.plot(ve, vv, marker='o', color='purple', label='Greedy policy on validation seeds')
    if best_episode:
        ax.scatter([best_episode], [best_score], s=140, color='gold', edgecolor='black', zorder=5, label='Best (kept)')
ax.set_title('Validation reward (no exploration)'); ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[2]
ax.plot(epsilon_per_episode, color='green', linewidth=2)
ax.fill_between(range(len(epsilon_per_episode)), epsilon_per_episode, alpha=0.2, color='green')
ax.set_title('Epsilon decay (per episode)'); ax.set_xlabel('Episode'); ax.set_ylabel('Epsilon'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/dqn_training.png', dpi=150, bbox_inches='tight')
plt.show()

## STEP 9 — Measured comparison: ITMS (DQN) vs fixed-time signals
Every policy faces the **same 30 random one-hour scenarios** (same seeds, never used in training or validation). These are the numbers the dashboard uses — nothing is hard-coded.

In [ ]:
# STEP 9: fair evaluation on identical scenarios
EVAL_SEEDS = list(range(20000, 20030))

policies = {f'Fixed {g}s': (lambda s, e, a=a: a) for a, g in enumerate(ACTIONS)}
policies['Rule-based*']  = heuristic_policy
policies['ITMS (DQN)']   = lambda s, e: agent.act(s, training=False)

rows = {name: evaluate(p, EVAL_SEEDS) for name, p in policies.items()}
df_eval = pd.DataFrame(rows, index=['reward', 'wait_s', 'veh_per_hr', 'avg_queue']).T

print(f'{"Policy":<14}{"Reward":>10}{"Wait/veh (s)":>14}{"Throughput (veh/h)":>20}{"Avg queue":>11}')
print('-' * 69)
for name, r in df_eval.iterrows():
    mark = '  ← ITMS' if name == 'ITMS (DQN)' else ''
    print(f'{name:<14}{r.reward:>10.1f}{r.wait_s:>14.1f}{r.veh_per_hr:>20.1f}{r.avg_queue:>11.1f}{mark}')
print('* rule-based = a hand-written reference that sees the same queues (not a learned model)')

fixed_names = [n for n in df_eval.index if n.startswith('Fixed')]
best_fixed  = df_eval.loc[fixed_names, 'reward'].idxmax()
itms, bf, f60 = df_eval.loc['ITMS (DQN)'], df_eval.loc[best_fixed], df_eval.loc['Fixed 60s']

def pct(new, old): return (new - old) / old * 100
print(f'\nITMS vs best fixed timing ({best_fixed}):')
print(f'   wait per vehicle : {bf.wait_s:6.1f}s → {itms.wait_s:6.1f}s  ({pct(itms.wait_s, bf.wait_s):+.1f}%)')
print(f'   throughput       : {bf.veh_per_hr:6.0f}  → {itms.veh_per_hr:6.0f}   ({pct(itms.veh_per_hr, bf.veh_per_hr):+.1f}%)')
print(f'ITMS vs typical static plan (Fixed 60s):')
print(f'   wait per vehicle : {f60.wait_s:6.1f}s → {itms.wait_s:6.1f}s  ({pct(itms.wait_s, f60.wait_s):+.1f}%)')
print(f'   throughput       : {f60.veh_per_hr:6.0f}  → {itms.veh_per_hr:6.0f}   ({pct(itms.veh_per_hr, f60.veh_per_hr):+.1f}%)')
print('\n' + ('✅ ITMS beats every fixed-time plan on reward.' if itms.reward > bf.reward else
              '⚠️ ITMS did NOT beat the best fixed plan - train longer (raise EPISODES) before reporting results.'))

df_eval.round(3).to_csv(f'{RESULTS_DIR}/evaluation.csv')
with open(f'{MODELS_DIR}/final_eval.json', 'w') as f:
    json.dump({'best_fixed': best_fixed, 'itms': itms.round(3).to_dict(), 'best_fixed_metrics': bf.round(3).to_dict(),
               'fixed_60s': f60.round(3).to_dict(), 'n_eval_scenarios': len(EVAL_SEEDS)}, f, indent=2)

## STEP 10 — Full ITMS decision pipeline
Each cycle: read the 4 detectors → build the **same** feature row the LSTM was trained on → forecast → build the **same** state the DQN was trained on → choose the green time → apply emergency/accident overrides.
(The LSTM was trained on the junction-1 *north* approach, so the forecast is for approach 0.)

In [ ]:
# STEP 10: run the full pipeline
def generate_detector_reading(ts, emergency_prob=0.08, accident_prob=0.02):
    hour = ts.hour + ts.minute / 60
    rs = [sample_reading(hour, d) for d in DIRECTIONS]       # same sensor model as the training data
    return {'timestamp': ts.isoformat(), 'hour': ts.hour, 'day_of_week': ts.weekday(),
            'intersection_id': 'karachi_j1',
            'vehicle_counts': [r['vehicle_count'] for r in rs], 'flow_rates': [r['flow_rate'] for r in rs],
            'queue_lengths': [r['queue_length'] for r in rs],   'occupancy': [r['occupancy'] for r in rs],
            'average_speeds': [r['average_speed'] for r in rs],
            'emergency_present': bool(np.random.random() < emergency_prob),
            'accident_detected': bool(np.random.random() < accident_prob)}

def make_feature_row(rd, appr=0):
    # exactly the 10 FEATURES the LSTM was trained on (6 scaled + 4 time encodings)
    c, q, occ, spd = rd['vehicle_counts'][appr], rd['queue_lengths'][appr], rd['occupancy'][appr], rd['average_speeds'][appr]
    cong = float(np.clip(occ * 0.4 + q / 50 * 0.4 + (1 - spd / 50) * 0.2, 0, 1))
    raw  = pd.DataFrame([[c, rd['flow_rates'][appr], occ, spd, q, cong]], columns=NUM_COLS)
    z    = scaler.transform(raw)[0]
    h, d = rd['hour'], rd['day_of_week']
    return np.array([*z, np.sin(2*np.pi*h/24), np.cos(2*np.pi*h/24),
                     np.sin(2*np.pi*d/7), np.cos(2*np.pi*d/7)], dtype=np.float32)

def run_itms_pipeline(rd, phase, last_green, progress, lstm_m, dqn_agent, window):
    queues = rd['queue_lengths']
    # 1. LSTM forecast for approach 0 (back to real vehicles)
    with tf.device(LSTM_DEVICE):
        pred = lstm_m(np.asarray(window[-LOOKBACK:], np.float32)[np.newaxis], training=False).numpy()[0]
    pred_vehicles = pred * VC_STD + VC_MEAN
    # 2. DQN decision - identical state layout to training
    state = make_state(queues, phase, last_green, progress)
    action_idx = dqn_agent.act(state, training=False)
    green = ACTIONS[action_idx]
    # 3. Safety overrides
    override = False
    if rd['emergency_present']:
        green, override = max(green, 90), True
    if rd['accident_detected']:
        green = min(green + 20, 120)
    return {'green_duration': green, 'action_idx': action_idx, 'emergency_override': override,
            'accident_mode': rd['accident_detected'], 'predicted_flow': float(np.mean(pred_vehicles)),
            'total_queue': sum(queues), 'total_vehicles': sum(rd['vehicle_counts'])}

N_CYCLES = 50
t_start  = datetime(2024, 1, 15, 7, 0, 0)
history  = []
for k in range(LOOKBACK):                                   # warm-up: fill the LSTM window with real readings
    history.append(make_feature_row(generate_detector_reading(t_start + timedelta(seconds=k * STEP_SEC))))

log, last_green = [], 0.0
print(f'Running {N_CYCLES} ITMS decision cycles...\n')
print(f'{"Cycle":>5} {"Serving":>8} {"Vehicles":>9} {"Queue":>6} {"Green":>6} {"Forecast":>9} {"Emergency":>10} {"Accident":>9}')
print('-' * 72)
for cycle in range(1, N_CYCLES + 1):
    ts = t_start + timedelta(seconds=(LOOKBACK + cycle) * STEP_SEC)
    reading = generate_detector_reading(ts)
    phase   = (cycle - 1) % 4
    result  = run_itms_pipeline(reading, phase, last_green, (cycle - 1) / N_CYCLES,
                                lstm_model, agent, history)
    last_green = float(result['green_duration'])
    log.append({**reading, **result, 'cycle': cycle, 'serving': DIRECTIONS[phase]})
    history.append(make_feature_row(reading))
    em = '🚨 YES' if result['emergency_override'] else 'No'
    ac = '⚠️ YES' if result['accident_mode'] else 'No'
    print(f'{cycle:>5} {DIRECTIONS[phase]:>8} {result["total_vehicles"]:>9} {result["total_queue"]:>6} '
          f'{result["green_duration"]:>5}s {result["predicted_flow"]:>9.1f} {em:>10} {ac:>9}')

df_log = pd.DataFrame(log)
df_log.drop(columns=['vehicle_counts', 'flow_rates', 'queue_lengths', 'occupancy', 'average_speeds']
            ).to_csv(f'{RESULTS_DIR}/pipeline_log.csv', index=False)

print('\n' + '=' * 60)
print('ITMS Pipeline Summary')
print(f'  Cycles run          : {N_CYCLES}')
print(f'  Avg green duration  : {df_log["green_duration"].mean():.1f} s')
print(f'  Avg vehicles/cycle  : {df_log["total_vehicles"].mean():.1f}')
print(f'  Avg queue/cycle     : {df_log["total_queue"].mean():.1f}')
print(f'  Avg LSTM forecast   : {df_log["predicted_flow"].mean():.1f} vehicles (approach 0)')
print(f'  Emergency overrides : {int(df_log["emergency_override"].sum())}')
print(f'  Accident events     : {int(df_log["accident_mode"].sum())}')
print('=' * 60)

## STEP 11 — Results dashboard (all values measured)

In [ ]:
# STEP 11: dashboard - every number comes from Steps 7 and 9
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('ITMS Results — measured in simulation\nITMS (DQN) vs fixed-time signal plans',
             fontsize=14, fontweight='bold')

names  = list(dict.fromkeys(['Fixed 60s', best_fixed, 'ITMS (DQN)']))
labels = [('Typical static\n(Fixed 60s)' if n == 'Fixed 60s' else
           f'Best fixed\n({n.replace("Fixed ", "")})' if n == best_fixed else 'ITMS\n(DQN)') for n in names]
cols   = ['#E74C3C' if n == 'Fixed 60s' else '#F39C12' if n == best_fixed else '#27AE60' for n in names]

def bar_panel(ax, metric, title, ylabel, fmt, lower_is_better):
    vals = [df_eval.loc[n, metric] for n in names]
    bars = ax.bar(labels, vals, color=cols, width=0.55, edgecolor='white', linewidth=1.5)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() * 1.01, fmt.format(v), ha='center', fontweight='bold')
    ref, new = df_eval.loc[best_fixed, metric], df_eval.loc['ITMS (DQN)', metric]
    ch = (new - ref) / ref * 100
    good = (ch < 0) if lower_is_better else (ch > 0)
    ax.set_title(f'{title}\nITMS vs best fixed: {ch:+.1f}%', fontweight='bold', color='green' if good else 'firebrick')
    ax.set_ylabel(ylabel); ax.set_ylim(0, max(vals) * 1.18); ax.grid(axis='y', alpha=0.3)

bar_panel(axes[0, 0], 'wait_s', 'Average wait per vehicle', 'Seconds', '{:.0f}s', True)
bar_panel(axes[0, 1], 'veh_per_hr', 'Intersection throughput', 'Vehicles / hour', '{:.0f}', False)

ax = axes[1, 0]
ax.hist(df_log['green_duration'], bins=np.arange(10, 135, 10), color='steelblue', edgecolor='white', rwidth=0.85)
ax.set_title(f'Green durations chosen in the {N_CYCLES}-cycle pipeline run', fontweight='bold')
ax.set_xlabel('Green duration (s)'); ax.set_ylabel('Count'); ax.grid(axis='y', alpha=0.3)

ax = axes[1, 1]
n_show = min(300, len(y_te))
ax.plot(y_te[:n_show, 0] * VC_STD + VC_MEAN, color='steelblue', linewidth=1.5, label='Actual')
ax.plot(y_pred[:n_show, 0] * VC_STD + VC_MEAN, color='red', linewidth=1.2, linestyle='--', label='LSTM (1 step ahead)')
ax.set_title(f'LSTM on unseen test data  (R² = {r2:.3f}, MAE = {mae_v:.1f} vehicles)', fontweight='bold')
ax.set_xlabel('Test sample'); ax.set_ylabel('Vehicles'); ax.legend(); ax.grid(alpha=0.3)

fig.text(0.5, 0.005, f'Simulated environment (TrafficEnv v2) · {len(EVAL_SEEDS)} one-hour scenarios, identical for every policy',
         ha='center', fontsize=9, style='italic')
plt.tight_layout(rect=[0, 0.02, 1, 1])
plt.savefig(f'{RESULTS_DIR}/final_results_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Dashboard saved to {RESULTS_DIR}/final_results_dashboard.png')

## STEP 12 — YOLOv8 vehicle-detection demo (optional, last on purpose)
Placed last so that installing `ultralytics` can never disturb the training steps above. Skipped automatically if Internet is OFF. It never stops the run.

In [ ]:
# STEP 12: optional YOLOv8 demo
def internet_ok(host='github.com', port=443, timeout=5):
    import socket
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False

def ensure_ultralytics():
    try:
        import ultralytics
        return True
    except ImportError:
        pass
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'],
                   capture_output=True, text=True, timeout=600)
    import importlib; importlib.invalidate_caches()
    import ultralytics
    return True

def run_yolo_demo():
    ensure_ultralytics()
    from ultralytics import YOLO
    import cv2
    VEHICLE = {2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}
    cwd = os.getcwd()
    os.chdir('/tmp')                                   # keep the ~50 MB weights out of the saved output
    try:
        print('Loading YOLOv8m (downloads ~50 MB on first use)...')
        model = YOLO('yolov8m.pt')
    finally:
        os.chdir(cwd)
    print('✅ YOLOv8m loaded')

    img = np.full((480, 640, 3), 80, np.uint8)         # synthetic road scene
    for x in range(0, 640, 60):
        cv2.rectangle(img, (x, 230), (x + 30, 250), (255, 255, 255), -1)
    boxes = [(50,150,90,50,(0,120,255)), (200,160,90,50,(255,80,0)), (370,145,90,50,(0,200,100)),
             (520,155,90,50,(150,0,200)), (100,300,130,65,(200,200,0)), (330,305,130,65,(0,180,200))]
    for x, y, w, h, col in boxes:
        cv2.rectangle(img, (x, y), (x + w, y + h), col, -1); cv2.rectangle(img, (x, y), (x + w, y + h), (0, 0, 0), 1)

    res = model(img, conf=0.5, iou=0.45, verbose=False)[0]          # single inference
    counts = {v: 0 for v in VEHICLE.values()}
    for b in res.boxes:
        if int(b.cls[0]) in VEHICLE:
            counts[VEHICLE[int(b.cls[0])]] += 1
    print(f'\nYOLOv8 on the synthetic image: {sum(counts.values())} vehicles {counts}')
    print('(coloured rectangles are not real cars, so 0 detections is expected - use a real road photo for real counts)')

    illus = img.copy()
    for (x, y, w, h, _), (lab, c) in zip(boxes, [('car',(0,200,0)), ('car',(0,200,0)), ('bus',(255,140,0)),
                                                 ('motorcycle',(0,180,220)), ('truck',(180,0,220)), ('car',(0,200,0))]):
        cv2.rectangle(illus, (x, y), (x + w, y + h), c, 2)
        cv2.putText(illus, lab, (x + 4, y + 16), cv2.FONT_HERSHEY_SIMPLEX, 0.5, c, 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)); axes[0].set_title('YOLOv8 actual output', fontweight='bold')
    axes[1].imshow(cv2.cvtColor(illus, cv2.COLOR_BGR2RGB));      axes[1].set_title('Illustration of box style (hand-drawn, NOT detections)', fontweight='bold')
    for a in axes: a.axis('off')
    plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/yolo_detection.png', dpi=150, bbox_inches='tight'); plt.show()

try:
    if internet_ok():
        run_yolo_demo()
    else:
        print('ℹ️ Internet is OFF - YOLO demo skipped (everything else is complete).')
except Exception as e:
    print(f'⚠️ YOLO demo skipped ({type(e).__name__}: {e}). Everything else is complete.')
finally:
    for stray in glob.glob(f'{WORK}/yolov8*.pt'):
        try: os.remove(stray)
        except OSError: pass

In [ ]:
# STEP 13: final summary
print('\n' + '=' * 62)
print('            ITMS PROJECT — COMPLETE SUMMARY')
print('=' * 62)
print(f'\n⏱️  Total notebook time: {elapsed_min():.1f} min')

print('\n📁 Saved files:')
for root, dirs, files in os.walk(WORK):
    if any(s in root for s in ['.git', '__pycache__', '.config', '.virtual_documents', 'lstm_backup']):
        continue
    for f in sorted(files):
        p = os.path.join(root, f); sz = os.path.getsize(p)
        print(f'   {os.path.relpath(p, WORK):<46}{(f"{sz/1024:.0f} KB" if sz > 1024 else f"{sz} B"):>10}')

print('\n📊 Metrics:')
for name, fn in [('LSTM', 'lstm_metrics.json'), ('DQN', 'dqn_metrics.json'), ('Eval', 'final_eval.json')]:
    p = f'{MODELS_DIR}/{fn}'
    if os.path.exists(p):
        print(f'   {name:<5}', json.load(open(p)))

print('\n' + '=' * 62)
if diverged:
    print('  ❌ DQN diverged - re-run (it resumes from the last good checkpoint)')
elif stopped_early:
    print('  ⚠️ DQN stopped early - add this version as Input and re-run to finish')
else:
    print('  ALL STEPS COMPLETE 🎉')
print('=' * 62)